# 1.) Classical Life Insurance Calculation

In [2]:
# Calculate annuity of term life insurance
def get_termlife_annuity(age, duration, interest, A= 0.00022, B=2.7*10**(-6), c=1.124):
    
    v = 1/(1+interest)
    ann = 0
    for k in range(duration):
        ann+= v**k*np.exp(-A*k-B/np.log(c)*c**(age)*(c**k-1))                  
    
    return ann

In [3]:
# Calculate apv of benefits for term life insurance
def get_termlife_APV_ben(age, duration, interest, A= 0.00022, B=2.7*10**(-6), c=1.124):
    
    v = 1/(1+interest)
    apv = 0
    for k in range(duration):
        apv += v**(k+1)*np.exp(-A*k-B/np.log(c)*c**(age)*(c**k-1))*(1-np.exp(-A-B/np.log(c)*c**(age+k)*(c-1)))
    
    return apv

In [4]:
# Calculate premium for term life insurance
def get_termlife_premium(age_init,Sum_ins,duration,  interest,  A= 0.00022, B=2.7*10**(-6), c=1.124):
    
    return Sum_ins*get_termlife_APV_ben(age_init, duration, interest, A, B, c)/get_termlife_annuity(age_init, duration, interest, A, B, c)

In [5]:
# Calculate current Policy value of term life insurance contract
def get_termlife_reserve(age_curr, Sum_ins, duration,  interest,age_of_contract=0, A= 0.00022, B=2.7*10**(-6), c=1.124):
    age_init = age_curr-age_of_contract
    prem = get_termlife_premium(age_init,Sum_ins, duration, interest, A,B,c)
    apv_prem = get_termlife_annuity(age_curr, duration-age_of_contract, interest, A,B,c)
    apv_ben = get_termlife_APV_ben(age_curr, duration-age_of_contract, interest, A,B,c)
    
    return Sum_ins*apv_ben -prem*apv_prem 

In [1]:
# Calculate Policy values up to maturity of term life insurance contract
# Potentially even including the past policy values, starting at the start of the contract
def get_termlife_reserve_profile(age_curr, Sum_ins, duration, interest,age_of_contract = 0,  A= 0.00022, 
                                 B=2.7*10**(-6), c=1.124, option_past = True, age_limit = 120):
    
    age_init = age_curr-age_of_contract
    premium = get_termlife_premium(age_init, Sum_ins,duration, interest, A,B,c)
    reserve = np.zeros(duration+1)
    
    # No expenses so far
    e_ann = 0
    e_init = 0
    # No claims-related expenses
    E_claim = 0
    reserve[0] = - e_init
    reserve[-1] = 0
    for k in range(duration-1): # Exclude value of reserve at maturity of contract (set 0, to avoid inaccuracy due to rounding errors)
        if age_init+k < age_limit:
            prob_live = np.exp(-A-B/np.log(c)*c**(age_init+k)*(c-1))# at age x+k
            reserve[k+1] =((reserve[k]+premium-e_ann)*(1+interest) - (Sum_ins+E_claim)*(1-prob_live))/prob_live
        else:
            #reserve[k+1] = Sum_ins/(1+interest)+(e_ann+e_init)/(1+interest)-premium*1
            reserve[k+1] = (Sum_ins+(e_ann+e_init))*get_termlife_APV_ben(age_init+k,duration,interest)-premium*get_termlife_annuity(age_init+k, duration, interest)
        pass
    
    
    if option_past==False:
        if age_of_contract <= duration:
            reserve = reserve[age_of_contract:]
        else:
            reserve = 0

    return reserve   

# 2. Pensions

In [2]:
# Output: Expected Reserve up to max age
# Inputs: Accumulated Fund, Current Salary, Salary scale/ factor, Contribution Rate, SUS-Model (A,B,c)
#         Rechnungszins (const.), max. Rentenalter, Frührente (%-Anteile für Alter 60-66)
# max. Alter not included as we assume fund to be paid as lump sum at pension age. 
# Annuitization of a lump sum can be considered as a seperate contract

def get_pension_reserve(fund_accum = 0, age = 40, salary = 1, salary_scale = 0.02, contribution = 0.03, 
                         A = 0.00022, B = 2.7*10**(-6), c = 1.124, interest = 0.01,
                        pension_age_max = 67,
                        early_pension = [0.1, 0.05, 0.05, 0.05, 0.05, 0.05]):
    
    #if len(early_pension) != 6:
     #   print('Check early_pension vector!')
      #  return
        
    reserve = np.zeros(pension_age_max-age+1)
    reserve[0] = fund_accum
    for i in range(pension_age_max-age-len(early_pension)-1):
        #print(age+i)
        reserve[i+1] = (reserve[i]++contribution*salary*(1+salary_scale)**(i))*(1+interest)*\
                        np.exp(-A-B/np.log(c)*c**(age+i)*(c-1))
        #reserve[i+1]= (reserve[i] +
         #               contribution*salary*(1+salary_scale)**(i))*(1+interest) -
          #            (1-np.exp(-A*0.5-B/np.log(c)*c**(age+i-0.5)*(c**0.5-1)))* \
           #            (reserve[i]+contribution*salary*(1+salary_scale)**(i-0.5)))/ \
            #            np.exp(-A-B/np.log(c)*c**(age+i)*(c-1))
    
    k=0
    for i in range(pension_age_max-age-len(early_pension)-1,pension_age_max-age-1):    
        
        #print(k), print(age+i), print(early_pension[k])
        # Old Reserve + Contribution - Mid-year exit in case of death 
        # or exit at beginning of year for early retirement
        reserve[i+1] = (reserve[i]++contribution*salary*(1+salary_scale)**(i))*(1+interest)*\
                        np.exp(-A-B/np.log(c)*c**(age+i)*(c-1))*(1-early_pension[k])
        #reserve[i+1] = ((reserve[i]+
         #                contribution*salary*(1+salary_scale)**(i))*(1+interest) -
          #            (1-early_pension[k])*(1-np.exp(-A*0.5-B/np.log(c)*c**(age+i-0.5)*(c**0.5-1)))* \
           #            (reserve[i]+contribution*salary*(1+salary_scale)**(i-0.5))- \
            #           early_pension[k]*reserve[i])/\
             #           np.exp(-A-B/np.log(c)*c**(age+i)*(c-1))/(1-early_pension[k])       
        k+=1
    
    return reserve


In [5]:
# matrix Version of get_pension_reserve
# input_used = ['Fund','Age', 'Salary', 'Salary_scale', 'Contribution', 'interest_rate']

def pension_reserve(data, pension_age_max = 67, age_min = 25, interest_std = 0.03, 
                    ep_structure = [0.1, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05]):
    
    length = pension_age_max - age_min + 1
    reserves = np.zeros([data.shape[0],length])
    
    if data.shape[1] == 6:
        for i in range(data.shape[0]):

            reserves[i,0:length - data[i,1].astype('int')+age_min] = get_pension_reserve( 
                                                                     age =data[i,1].astype('int'),
                                                                     fund_accum = data[i,0], 
                                                                     salary = data[i,2], salary_scale = data[i,3], 
                                                                     contribution = data[i,4], interest = data[i,5],
                                                                     pension_age_max = 67,
                                                                     early_pension = ep_structure)
    elif data.shape[1] == 5:
        
        for i in range(data.shape[0]):

            reserves[i,0:length - data[i,1].astype('int')+age_min] = get_pension_reserve(
                                                             age =data[i,1].astype('int'), 
                                                             fund_accum = data[i,0], 
                                                             salary = data[i,2], salary_scale = data[i,3], 
                                                             contribution = data[i,4], interest = interest_std,
                                                             pension_age_max = 67,
                                                             early_pension = ep_structure)
    
    
    return reserves